# Round 4 agent2 alpha hunt

Goal: find **specific**, high-conviction bot/tape patterns and convert only the strongest into trader logic.

Focus:
1. Bot+product+side lead/lag edges.
2. Cross-product flow (wing vouchers -> `VEV_5200`).
3. Threshold sweeps for signal quality.

In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("../..").resolve()
DATA = ROOT / "data" / "ROUND_4"
DAYS = (1, 2, 3)

trades = pd.concat(
    [pd.read_csv(DATA / f"trades_round_4_day_{d}.csv", sep=";").assign(day=d) for d in DAYS],
    ignore_index=True,
)
prices = pd.concat(
    [pd.read_csv(DATA / f"prices_round_4_day_{d}.csv", sep=";").assign(day=d) for d in DAYS],
    ignore_index=True,
)
print("trades:", len(trades), "prices:", len(prices))
print("bots:", sorted(set(trades["buyer"]) | set(trades["seller"])))

In [ ]:
def build_mid_index(product: str):
    m = (
        prices.loc[prices["product"] == product, ["day", "timestamp", "mid_price"]]
        .sort_values(["day", "timestamp"])
    )
    return {d: g[["timestamp", "mid_price"]].to_numpy() for d, g in m.groupby("day")}


def move(index, day: int, ts: int, h: int = 100):
    arr = index.get(day)
    if arr is None:
        return None
    t = arr[:, 0]
    m = arr[:, 1]
    i = np.searchsorted(t, ts, side="left")
    j = np.searchsorted(t, ts + h, side="left")
    if i >= len(t) or j >= len(t):
        return None
    return float(m[j] - m[i])


idx_5200 = build_mid_index("VEV_5200")
idx_velvet = build_mid_index("VELVETFRUIT_EXTRACT")

In [ ]:
# Per-product, per-bot, per-side signed markout at h=100.
def side_markout(symbol: str, h: int = 100, min_n: int = 20):
    idx = build_mid_index(symbol)
    sub = trades[trades["symbol"] == symbol]
    rows = []
    bots = sorted(set(sub["buyer"]) | set(sub["seller"]))
    for bot in bots:
        for side in ("buy", "sell"):
            ev = sub[sub["buyer" if side == "buy" else "seller"] == bot]
            if len(ev) < min_n:
                continue
            vals = []
            for r in ev.itertuples(index=False):
                mv = move(idx, int(r.day), int(r.timestamp), h)
                if mv is None:
                    continue
                vals.append(mv if side == "buy" else -mv)
            if len(vals) < min_n:
                continue
            rows.append({
                "symbol": symbol,
                "bot": bot,
                "side": side,
                "n": len(vals),
                "mean_h100": float(np.mean(vals)),
                "hit_h100": float(np.mean(np.array(vals) > 0)),
            })
    return pd.DataFrame(rows)


for s in ["VELVETFRUIT_EXTRACT", "HYDROGEL_PACK", "VEV_5200", "VEV_5300", "VEV_5400"]:
    out = side_markout(s)
    print("\n==", s, "==")
    display(out.sort_values("mean_h100", ascending=False).head(8))

In [ ]:
# Specific alpha: wing flow predicts VEV_5200 direction.
WING_PRODUCTS = ["VEV_5300", "VEV_5400", "VEV_5500"]
alpha = {"Mark 22": -1.0, "Mark 01": +1.0, "Mark 14": +0.5}

wing = trades[trades["symbol"].isin(WING_PRODUCTS)].copy()
wing["score"] = wing["quantity"] * (
    wing["buyer"].map(alpha).fillna(0.0) - wing["seller"].map(alpha).fillna(0.0)
)
agg = wing.groupby(["day", "timestamp"], as_index=False)["score"].sum()

pairs = []
for r in agg.itertuples(index=False):
    mv = move(idx_5200, int(r.day), int(r.timestamp), 100)
    if mv is not None:
        pairs.append((float(r.score), mv))
pairs = np.array(pairs)

rows = []
for th in [5, 8, 10, 12, 15, 20]:
    for sign, label in [(+1, "up"), (-1, "down")]:
        sel = pairs[:, 0] >= th if sign > 0 else pairs[:, 0] <= -th
        n = int(sel.sum())
        if n < 20:
            continue
        signed = pairs[sel, 1] * (1 if sign > 0 else -1)
        rows.append({
            "threshold": th,
            "direction": label,
            "n": n,
            "mean_signed_move_h100": float(np.mean(signed)),
            "hit_rate": float(np.mean(signed > 0)),
        })
display(pd.DataFrame(rows).sort_values(["mean_signed_move_h100", "hit_rate"], ascending=False))

## Agent2 takeaway

- **Biggest new deployable edge**: wing bot flow (`Mark 22` vs `Mark 01`) predicts `VEV_5200` in the next ~100 ticks.
- Implemented as an overlay only on `VEV_5200` target selection to avoid contaminating the rest of the stack.
- Best tested variant in `ROUND_4/traders/agent2` is the aggressive one (`_C`), then promoted to main.

In [ ]:
print("Rank command:")
print("uv run rank --show-per-product --trader trader_HYPER.py --trader traders/agent2/trader_AGENT2_WING5200_C.py --no-cache")
print("Overfit command:")
print("uv run check_overfit --all --by-pnl")